# Block 16A — ROI-only UAV DSM structure audit
**Musanga / NB06 companion.** Uses only complete 50 × 50 m windows inside `ROI.shp`. Does not alter NB06 or train models. DSM files read from `Final/Data/DSM`, outputs to `CLEAN_NB06/16_DSM_structure/ROI_only`.
The master NB06 notebook could not be safely modified because its full content was unavailable through the GitHub connector.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import time
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from shapely.geometry import box
from IPython.display import display

T0=time.time()
BASE=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DSM_DIR=BASE/'Data'/'DSM'
OUT=BASE/'Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06/16_DSM_structure/ROI_only'
OUT.mkdir(parents=True,exist_ok=True)
roi_paths=list((BASE/'Data').rglob('ROI.shp'))
if not roi_paths: roi_paths=list(BASE.rglob('ROI.shp'))
if len(roi_paths)!=1: raise RuntimeError(f'Expected one ROI.shp, found {len(roi_paths)}: {roi_paths}')
roi=gpd.read_file(roi_paths[0])
assert roi.crs is not None and not roi.empty
roi=roi.loc[roi.geometry.notna() & ~roi.geometry.is_empty].copy().reset_index(drop=True)
roi['roi_id']=np.arange(len(roi))
roi['geometry']=roi.geometry.make_valid()
print('ROI:',roi_paths[0], '| polygons:',len(roi))
files=sorted(p for p in DSM_DIR.rglob('*') if p.suffix.lower() in ('.tif','.tiff'))
assert files,'No DSM files found'
WINDOW_M=50.; STRIDE_M=25.; GRID_N=50; MIN_VALID=.90; MAX_PER_DSM=120
rng=np.random.default_rng(1601)

def surface_metrics(z):
    z=np.asarray(z,float)
    good=np.isfinite(z)
    if good.mean()<MIN_VALID: return None
    yy,xx=np.indices(z.shape)
    a=np.column_stack([xx[good],yy[good],np.ones(good.sum())])
    beta=np.linalg.lstsq(a,z[good],rcond=None)[0]
    resid=z-(beta[0]*xx+beta[1]*yy+beta[2])
    xok=good[:,1:] & good[:,:-1]
    yok=good[1:,:] & good[:-1,:]
    delta=np.r_[(resid[:,1:]-resid[:,:-1])[xok],(resid[1:,:]-resid[:-1,:])[yok]]
    if not len(delta): return None
    v=resid[good]
    return dict(valid_fraction=float(good.mean()),
       detrended_sd_m=float(v.std()),
       detrended_iqr_m=float(np.percentile(v,75)-np.percentile(v,25)),
       detrended_p90_p10_m=float(np.percentile(v,90)-np.percentile(v,10)),
       local_difference_m=float(np.median(np.abs(delta))),
       mean_surface_elevation_m=float(np.mean(z[good])))

rows=[]; coverage=[]
for path in files:
    start=time.time()
    with rasterio.open(path) as ds:
        assert ds.crs and ds.crs.is_projected, f'Projected DSM needed: {path.name}'
        polys=roi.to_crs(ds.crs)
        extent=box(*ds.bounds)
        candidates=[]
        for _,r in polys.iterrows():
            poly=r.geometry
            if not poly.intersects(extent): continue
            xmin,ymin,xmax,ymax=poly.bounds
            xs=np.arange(np.ceil((xmin+WINDOW_M/2)/STRIDE_M)*STRIDE_M,xmax-WINDOW_M/2+1e-6,STRIDE_M)
            ys=np.arange(np.ceil((ymin+WINDOW_M/2)/STRIDE_M)*STRIDE_M,ymax-WINDOW_M/2+1e-6,STRIDE_M)
            for x in xs:
                for y in ys:
                    cell=box(x-WINDOW_M/2,y-WINDOW_M/2,x+WINDOW_M/2,y+WINDOW_M/2)
                    if poly.covers(cell) and extent.covers(cell): candidates.append((int(r.roi_id),float(x),float(y)))
        candidates=list(dict.fromkeys(candidates))
        rng.shuffle(candidates)
        selected=candidates[:MAX_PER_DSM]
        coverage.append(dict(file=path.name,eligible_windows=len(candidates),selected_windows=len(selected),nodata=ds.nodata))
        print('START',path.name,'eligible',len(candidates),'selected',len(selected),flush=True)
        accepted=0
        for roi_id,x,y in selected:
            win=from_bounds(x-WINDOW_M/2,y-WINDOW_M/2,x+WINDOW_M/2,y+WINDOW_M/2,transform=ds.transform)
            raw=ds.read(1,window=win,out_shape=(GRID_N,GRID_N),masked=True,resampling=Resampling.bilinear)
            z=np.asarray(raw.astype('float64').filled(np.nan))
            z[(z==0)|(z==-10000)]=np.nan
            info=surface_metrics(z)
            if info is None: continue
            rows.append(dict(file=path.name,roi_id=roi_id,center_x=x,center_y=y,**info))
            accepted+=1
        print('DONE',path.name,'accepted',accepted,'seconds',round(time.time()-start,1),flush=True)

pd.DataFrame(coverage).to_csv(OUT/'16A_ROI_only_inventory.csv',index=False)
display(pd.DataFrame(coverage))
metrics=pd.DataFrame(rows)
if metrics.empty: raise RuntimeError('No valid fully enclosed 50m DSM windows; do not sample outside ROI')
metrics.to_csv(OUT/'16A_ROI_only_50m_metrics.csv',index=False)
summary=metrics.groupby(['file','roi_id'],as_index=False).agg(
    n=('detrended_sd_m','size'),
    median_roughness_m=('detrended_sd_m','median'),
    median_iqr_m=('detrended_iqr_m','median'),
    median_local_difference_m=('local_difference_m','median'),
    p90_roughness_m=('detrended_sd_m',lambda x:x.quantile(.9)))
summary.to_csv(OUT/'16A_ROI_only_summary_by_polygon.csv',index=False)
display(summary.round(3))
print('FINISHED in',round(time.time()-T0,1),'s. Outputs:',OUT)
print('Surface roughness is not ground-normalized canopy height. Overlapping windows are not independent.')